In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostRegressor

In [2]:
comp_df = pd.read_csv('train.csv')
original_df = pd.read_csv('synthetic_road_accidents_100k.csv')
test_df = pd.read_csv('test.csv')

# Saved the test IDs now so we don't lose them
test_ids = test_df['id']

In [3]:
# Combine the training data
comp_df = comp_df.drop('id', axis=1) 
comp_df['is_original'] = 0
original_df['is_original'] = 1
full_train_df = pd.concat([comp_df, original_df], ignore_index=True)

# List of our dataframes to apply feature engineering to
datasets = [full_train_df, test_df]

# Applying the same transformations to both train and test data
for df in datasets:
    # credit goes to @tilii for this feature
    df['gm_risk_feature'] = (
        0.3 * df["curvature"] + 0.2 * (df["lighting"] == "night").astype(int) +
        0.1 * (df["weather"] != "clear").astype(int) + 0.2 * (df["speed_limit"] >= 60).astype(int) +
        0.1 * (np.array(df["num_reported_accidents"]) > 2).astype(int)
    )
    
    # Convert boolean columns to integers for consistency
    for col in df.select_dtypes(include='bool').columns:
        df[col] = df[col].astype(int)

print("Feature Engineering Complete!")

Feature Engineering Complete!


In [4]:
# Separate target variable from the full training data
X = full_train_df.drop('accident_risk', axis=1)
y = full_train_df['accident_risk']

# Drop the id from the test features (we already saved test_ids)
test_features = test_df.drop('id', axis=1)

categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("--- Training Data After Encoding ---")
print(X.head())

test_features = pd.get_dummies(test_features, columns=categorical_cols, drop_first=True)

print("--- Test Data After Encoding ---")
print(test_features.head())

X_aligned, test_aligned = X.align(test_features, join='left', axis=1, fill_value=0)

--- Training Data After Encoding ---
   num_lanes  curvature  speed_limit  road_signs_present  public_road  \
0          2       0.06           35                   0            1   
1          4       0.99           35                   1            0   
2          4       0.63           70                   0            1   
3          4       0.07           35                   1            1   
4          1       0.58           60                   0            0   

   holiday  school_season  num_reported_accidents  is_original  \
0        0              1                       1            0   
1        1              1                       0            0   
2        1              0                       2            0   
3        0              0                       1            0   
4        1              0                       1            0   

   gm_risk_feature  road_type_rural  road_type_urban  lighting_dim  \
0            0.118            False             True     

In [5]:
for col in X_aligned.select_dtypes(include='bool').columns:
    X_aligned[col] = X_aligned[col].astype(int)
    test_aligned[col] = test_aligned[col].astype(int)

X_aligned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617754 entries, 0 to 617753
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   num_lanes               617754 non-null  int64  
 1   curvature               617754 non-null  float64
 2   speed_limit             617754 non-null  int64  
 3   road_signs_present      617754 non-null  int32  
 4   public_road             617754 non-null  int32  
 5   holiday                 617754 non-null  int32  
 6   school_season           617754 non-null  int32  
 7   num_reported_accidents  617754 non-null  int64  
 8   is_original             617754 non-null  int64  
 9   gm_risk_feature         617754 non-null  float64
 10  road_type_rural         617754 non-null  int32  
 11  road_type_urban         617754 non-null  int32  
 12  lighting_dim            617754 non-null  int32  
 13  lighting_night          617754 non-null  int32  
 14  weather_foggy       

In [7]:
from sklearn.model_selection import KFold

models_to_validate = {
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42, verbosity=-1), 
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0) 
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_cv_scores = {}

print("\n--- Running Baseline Cross-Validation ---")
for model_name, model in models_to_validate.items():
    print(f"\n--- Validating: {model_name} ---")
    
    model_scores = [] # Stores scores for the 5 folds of this model
    
    for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
        X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        
        # Train the current model
        model.fit(X_train, y_train)
        
        # Make predictions and calculate the score
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        model_scores.append(rmse)
        
    # Calculate the average score for this model across the 5 folds
    avg_rmse = np.mean(model_scores)
    all_cv_scores[model_name] = avg_rmse

print("\n--- FINAL BASELINE CV SCORE SUMMARY ---")
for model_name, score in all_cv_scores.items():
    print(f"{model_name}: {score}")


--- Running Baseline Cross-Validation ---

--- Validating: XGBoost ---

--- Validating: LightGBM ---

--- Validating: CatBoost ---

--- FINAL BASELINE CV SCORE SUMMARY ---
XGBoost: 0.05514747202591923
LightGBM: 0.05522618757325689
CatBoost: 0.055083999342601976


In [8]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# --- 1. Define the Objective Functions for Each Model ---
# Each function tells Optuna how to test a specific model.

def xgb_objective(trial, X_train, y_train, X_val, y_val):
    params = {
        'objective': 'reg:squarederror', 'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3), 'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0), 'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    model = XGBRegressor(**params).fit(X_train, y_train)
    preds = model.predict(X_val)
    return root_mean_squared_error(y_val, preds)

def lgbm_objective(trial, X_train, y_train, X_val, y_val):
    params = {
        'objective': 'regression_l1', 'metric': 'rmse', 'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300), 'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100), 'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0), 'random_state': 42
    }
    model = LGBMRegressor(**params).fit(X_train, y_train)
    preds = model.predict(X_val)
    return root_mean_squared_error(y_val, preds)

def catboost_objective(trial, X_train, y_train, X_val, y_val):
    params = {
        'objective': 'RMSE', 'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3), 'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 100.0, log=True),
        'bootstrap_type': 'Bayesian', 'random_state': 42, 'verbose': 0
    }
    model = CatBoostRegressor(**params).fit(X_train, y_train)
    preds = model.predict(X_val)
    return root_mean_squared_error(y_val, preds)

In [9]:
# --- 2. Set up the Tuning Loop ---

# Create a dictionary mapping model names to their objective functions
objective_functions = {
    "XGBoost": xgb_objective,
    "LightGBM": lgbm_objective,
    "CatBoost": catboost_objective
}

# Create a dictionary to store the best parameters for each model
best_params_all_models = {}

# Split data once for all tuning experiments
X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(X_aligned, y, test_size=0.2, random_state=42)

# The main loop that tunes each model
print("--- Starting Automated Hyperparameter Tuning ---")
for model_name, objective_func in objective_functions.items():
    print(f"\n--- Tuning {model_name} ---")
    
    # Create the Optuna study
    study = optuna.create_study(direction='minimize')
    
    # Run the optimization
    study.optimize(lambda trial: objective_func(trial, X_train_tune, y_train_tune, X_val_tune, y_val_tune), n_trials=25)
    
    # Store the best parameters
    best_params_all_models[model_name] = study.best_params
    print(f"Finished tuning {model_name}. Best RMSE: {study.best_value}")

# --- 3. Final Report ---
print("\n\n--- FINAL TUNING SUMMARY ---")
for model_name, params in best_params_all_models.items():
    print(f"\n--- Best Params for {model_name} ---")
    print(params)

[I 2025-10-24 21:39:17,023] A new study created in memory with name: no-name-9e7efdb5-fdc8-42db-addc-24b85a6f900b


--- Starting Automated Hyperparameter Tuning ---

--- Tuning XGBoost ---


[I 2025-10-24 21:39:38,626] Trial 0 finished with value: 0.056153691574669574 and parameters: {'n_estimators': 340, 'learning_rate': 0.025117366105561492, 'max_depth': 5, 'subsample': 0.6229133254142839, 'colsample_bytree': 0.611641441481991}. Best is trial 0 with value: 0.056153691574669574.
[I 2025-10-24 21:40:08,947] Trial 1 finished with value: 0.05609574466693057 and parameters: {'n_estimators': 722, 'learning_rate': 0.06753042581119893, 'max_depth': 3, 'subsample': 0.7172073253230983, 'colsample_bytree': 0.6101783717628523}. Best is trial 1 with value: 0.05609574466693057.
[I 2025-10-24 21:40:31,825] Trial 2 finished with value: 0.05566537840763594 and parameters: {'n_estimators': 451, 'learning_rate': 0.2105753752488744, 'max_depth': 5, 'subsample': 0.7807595628193118, 'colsample_bytree': 0.648690849977106}. Best is trial 2 with value: 0.05566537840763594.
[I 2025-10-24 21:41:11,704] Trial 3 finished with value: 0.05596583849911267 and parameters: {'n_estimators': 696, 'learning

Finished tuning XGBoost. Best RMSE: 0.05556044585957675

--- Tuning LightGBM ---


[I 2025-10-24 21:55:39,338] Trial 0 finished with value: 0.05590407348667313 and parameters: {'n_estimators': 531, 'learning_rate': 0.271819344260733, 'num_leaves': 24, 'max_depth': 12, 'min_child_samples': 34, 'subsample': 0.9921083223901344, 'colsample_bytree': 0.9157214980726708}. Best is trial 0 with value: 0.05590407348667313.
[I 2025-10-24 21:56:30,612] Trial 1 finished with value: 0.05598140346116956 and parameters: {'n_estimators': 600, 'learning_rate': 0.11501001817036524, 'num_leaves': 275, 'max_depth': 9, 'min_child_samples': 46, 'subsample': 0.8024210190573656, 'colsample_bytree': 0.812977782092799}. Best is trial 0 with value: 0.05590407348667313.
[I 2025-10-24 21:56:56,177] Trial 2 finished with value: 0.05600788180439189 and parameters: {'n_estimators': 550, 'learning_rate': 0.11463235351888469, 'num_leaves': 171, 'max_depth': 4, 'min_child_samples': 75, 'subsample': 0.6108447185396962, 'colsample_bytree': 0.9203639660848328}. Best is trial 0 with value: 0.05590407348667

Finished tuning LightGBM. Best RMSE: 0.05586858411239588

--- Tuning CatBoost ---


[I 2025-10-24 22:07:47,570] Trial 0 finished with value: 0.05624528726798253 and parameters: {'iterations': 768, 'learning_rate': 0.2445595484914702, 'depth': 9, 'l2_leaf_reg': 6.511016052946243e-08}. Best is trial 0 with value: 0.05624528726798253.
[I 2025-10-24 22:08:01,078] Trial 1 finished with value: 0.05633557573144215 and parameters: {'iterations': 189, 'learning_rate': 0.05934458515239378, 'depth': 4, 'l2_leaf_reg': 1.7333703576517325e-07}. Best is trial 0 with value: 0.05624528726798253.
[I 2025-10-24 22:08:37,672] Trial 2 finished with value: 0.05591279452627946 and parameters: {'iterations': 455, 'learning_rate': 0.05212167866002961, 'depth': 5, 'l2_leaf_reg': 1.2586658391064893e-07}. Best is trial 2 with value: 0.05591279452627946.
[I 2025-10-24 22:09:39,032] Trial 3 finished with value: 0.05569219561394101 and parameters: {'iterations': 754, 'learning_rate': 0.05376225498143548, 'depth': 6, 'l2_leaf_reg': 7.792032880829964e-05}. Best is trial 3 with value: 0.05569219561394

Finished tuning CatBoost. Best RMSE: 0.05560041271652856


--- FINAL TUNING SUMMARY ---

--- Best Params for XGBoost ---
{'n_estimators': 535, 'learning_rate': 0.044722518112694544, 'max_depth': 6, 'subsample': 0.6741670280631981, 'colsample_bytree': 0.9313065314625134}

--- Best Params for LightGBM ---
{'n_estimators': 263, 'learning_rate': 0.14559313313923916, 'num_leaves': 184, 'max_depth': 7, 'min_child_samples': 16, 'subsample': 0.8430754435749344, 'colsample_bytree': 0.7427433009426779}

--- Best Params for CatBoost ---
{'iterations': 750, 'learning_rate': 0.14477334610250991, 'depth': 6, 'l2_leaf_reg': 4.5482756485872766e-05}


In [10]:

# --- 1. Define our "Modular Toolbox" of TUNED models ---
# We are using the best parameters we just found from our Optuna tuning.

# Best parameters from your previous run
best_xgb_params = {'n_estimators': 535, 'learning_rate': 0.0447, 'max_depth': 6, 'subsample': 0.674, 'colsample_bytree': 0.931, 'random_state': 42}
best_lgbm_params = {'n_estimators': 263, 'learning_rate': 0.1455, 'num_leaves': 184, 'max_depth': 7, 'min_child_samples': 16, 'subsample': 0.843, 'colsample_bytree': 0.742, 'verbosity': -1, 'random_state': 42}
best_cat_params = {'iterations': 750, 'learning_rate': 0.1447, 'depth': 6, 'l2_leaf_reg': 4.548e-05, 'random_state': 42, 'verbose': 0}


models_to_validate = {
    "Tuned XGBoost": XGBRegressor(**best_xgb_params),
    "Tuned LightGBM": LGBMRegressor(**best_lgbm_params),
    "Tuned CatBoost": CatBoostRegressor(**best_cat_params)
}

# --- 2. Set up the 5-Fold Cross-Validation ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_cv_scores = {}

# --- 3. The "Testing Robot" that validates each model ---
print("\n--- Running Final Cross-Validation for Tuned Models ---")
for model_name, model in models_to_validate.items():
    print(f"\n--- Validating: {model_name} ---")
    model_scores = []
    
    for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
        X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        model_scores.append(rmse)
        
    avg_rmse = np.mean(model_scores)
    all_cv_scores[model_name] = avg_rmse

# --- 4. Final Report ---
print("\n\n--- FINAL TUNED CV SCORE SUMMARY ---")
# Let's sort the results to see the winner clearly
sorted_scores = sorted(all_cv_scores.items(), key=lambda item: item[1])

for model_name, score in sorted_scores:
    print(f"{model_name}: {score}")


--- Running Final Cross-Validation for Tuned Models ---

--- Validating: Tuned XGBoost ---

--- Validating: Tuned LightGBM ---

--- Validating: Tuned CatBoost ---


--- FINAL TUNED CV SCORE SUMMARY ---
Tuned CatBoost: 0.05509335148657131
Tuned XGBoost: 0.05512053590366228
Tuned LightGBM: 0.05516732097761805


In [11]:
# ==============================================================================
# ULTIMATE SUBMISSION: THE "DREAM TEAM" BLEND
# ==============================================================================

# 1. Use the best parameters we found for all three models
best_xgb_params = {'n_estimators': 535, 'learning_rate': 0.0447, 'max_depth': 6, 'subsample': 0.674, 'colsample_bytree': 0.931, 'random_state': 42}
best_lgbm_params = {'n_estimators': 263, 'learning_rate': 0.1455, 'num_leaves': 184, 'max_depth': 7, 'min_child_samples': 16, 'subsample': 0.843, 'colsample_bytree': 0.742, 'verbosity': -1, 'random_state': 42}
best_cat_params = {'iterations': 750, 'learning_rate': 0.1447, 'depth': 6, 'l2_leaf_reg': 4.548e-05, 'random_state': 42, 'verbose': 0}

# 2. Define our final models
final_xgb = XGBRegressor(**best_xgb_params)
final_lgbm = LGBMRegressor(**best_lgbm_params)
final_cat = CatBoostRegressor(**best_cat_params)

# 3. Train all three models on the ENTIRE dataset
print("--- Training Final Tuned Models on All Data ---")
final_xgb.fit(X_aligned, y)
final_lgbm.fit(X_aligned, y)
final_cat.fit(X_aligned, y)
print("Training complete!")

# 4. Make predictions on the test set
xgb_preds = final_xgb.predict(test_aligned)
lgbm_preds = final_lgbm.predict(test_aligned)
cat_preds = final_cat.predict(test_aligned)

# 5. Blend the predictions with our new 40/30/30 weighted average
final_blended_preds = (cat_preds * 0.4) + (xgb_preds * 0.3) + (lgbm_preds * 0.3)

# 6. Create the final submission file
submission_df = pd.DataFrame({'id': test_ids, 'accident_risk': final_blended_preds})
submission_df.to_csv('ultimate_team_submission.csv', index=False)

print("\nUltimate 'Dream Team' submission file created successfully!")

--- Training Final Tuned Models on All Data ---
Training complete!

Ultimate 'Dream Team' submission file created successfully!


In [12]:
# ==============================================================================
# FINAL EXPERIMENT: STACKING ENSEMBLE VALIDATION
# ==============================================================================
from sklearn.linear_model import Ridge

# (Your best_xgb_params, best_lgbm_params, and best_cat_params should be defined)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
stacking_scores = []

print("--- Starting Stacking Cross-Validation ---")

for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
    print(f"--- Fold {fold+1} ---")
    
    # Split data for this fold
    X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # --- LEVEL 0: Train our three base models ---
    xgb_model = XGBRegressor(**best_xgb_params).fit(X_train, y_train)
    lgbm_model = LGBMRegressor(**best_lgbm_params).fit(X_train, y_train)
    cat_model = CatBoostRegressor(**best_cat_params).fit(X_train, y_train)
    
    # --- Create features for the meta-model ---
    # These are the "Out-of-Fold" (OOF) predictions on the validation set
    oof_xgb = xgb_model.predict(X_val)
    oof_lgbm = lgbm_model.predict(X_val)
    oof_cat = cat_model.predict(X_val)
    
    # Stack these predictions into a new feature set for our meta-model
    meta_features = np.column_stack((oof_xgb, oof_lgbm, oof_cat))
    
    # --- LEVEL 1: Train our meta-model (the "Chief Strategist") ---
    # The meta-model learns from the predictions of the base models
    meta_model = Ridge(random_state=42)
    meta_model.fit(meta_features, y_val)
    
    # --- Final Prediction and Scoring for this fold ---
    # We use the trained meta-model to make the final prediction
    final_preds = meta_model.predict(meta_features)
    rmse = root_mean_squared_error(y_val, final_preds)
    stacking_scores.append(rmse)
    print(f"Stacking RMSE for Fold {fold+1}: {rmse}")

print(f"\nAverage CV RMSE for Stacking: {np.mean(stacking_scores)}")

--- Starting Stacking Cross-Validation ---
--- Fold 1 ---
Stacking RMSE for Fold 1: 0.05554892666454026
--- Fold 2 ---
Stacking RMSE for Fold 2: 0.054982463363787135
--- Fold 3 ---
Stacking RMSE for Fold 3: 0.05497585233784058
--- Fold 4 ---
Stacking RMSE for Fold 4: 0.05502378197612268
--- Fold 5 ---
Stacking RMSE for Fold 5: 0.05483562605407553

Average CV RMSE for Stacking: 0.055073330079273244


In [13]:
# ==============================================================================
# ULTIMATE SUBMISSION: THE FINAL STACKING ENSEMBLE
# ==============================================================================
from sklearn.linear_model import Ridge

# (Your best_xgb_params, best_lgbm_params, and best_cat_params should be defined)
best_xgb_params = {'n_estimators': 535, 'learning_rate': 0.0447, 'max_depth': 6, 'subsample': 0.674, 'colsample_bytree': 0.931, 'random_state': 42}
best_lgbm_params = {'n_estimators': 263, 'learning_rate': 0.1455, 'num_leaves': 184, 'max_depth': 7, 'min_child_samples': 16, 'subsample': 0.843, 'colsample_bytree': 0.742, 'verbosity': -1, 'random_state': 42}
best_cat_params = {'iterations': 750, 'learning_rate': 0.1447, 'depth': 6, 'l2_leaf_reg': 4.548e-05, 'random_state': 42, 'verbose': 0}


# --- Level 0: Train Base Models on ALL data ---
print("--- Training Final Base Models on All Data ---")
final_xgb = XGBRegressor(**best_xgb_params).fit(X_aligned, y)
final_lgbm = LGBMRegressor(**best_lgbm_params).fit(X_aligned, y)
final_cat = CatBoostRegressor(**best_cat_params).fit(X_aligned, y)
print("Base model training complete!")

# --- Create Meta-Features ---
# Predict on the training set to create training data for the meta-model
oof_preds_xgb = final_xgb.predict(X_aligned)
oof_preds_lgbm = final_lgbm.predict(X_aligned)
oof_preds_cat = final_cat.predict(X_aligned)
meta_features_train = np.column_stack((oof_preds_xgb, oof_preds_lgbm, oof_preds_cat))

# Predict on the test set to create the test data for the meta-model
test_preds_xgb = final_xgb.predict(test_aligned)
test_preds_lgbm = final_lgbm.predict(test_aligned)
test_preds_cat = final_cat.predict(test_aligned)
meta_features_test = np.column_stack((test_preds_xgb, test_preds_lgbm, test_preds_cat))

# --- Level 1: Train Meta-Model ---
print("\n--- Training Final Meta-Model ---")
meta_model = Ridge(random_state=42)
meta_model.fit(meta_features_train, y)
print("Meta-model training complete!")

# --- Final Prediction ---
# Use the trained meta-model to make final predictions
final_predictions = meta_model.predict(meta_features_test)

# --- Create Submission File ---
submission_df = pd.DataFrame({'id': test_ids, 'accident_risk': final_predictions})
submission_df.to_csv('ultimate_stacking_submission.csv', index=False)

print("\nUltimate stacking submission file created successfully!")

--- Training Final Base Models on All Data ---
Base model training complete!

--- Training Final Meta-Model ---
Meta-model training complete!

Ultimate stacking submission file created successfully!
